# Final Held-Out Test-Set Evaluation

> **TEST-SET USAGE POLICY.** This notebook is the ONLY place in this project where test-set labels are used. Every notebook before this one (01-04) loaded the test split and immediately discarded it. By this point every modeling decision — hyperparameters, calibration method per model, referral-rate grid, decision threshold — is already frozen from train/validation data. This notebook only **computes** metrics against those frozen choices; it never searches over alternatives using test performance. See `src/evaluate_final.py`'s module docstring for the full policy statement.
>
> **No value in any output table is simulated, guessed, or fabricated — every number is computed directly from saved test predictions.** Where a metric is undefined for the data at hand (e.g. a subgroup too small, or containing only one class), the result is `NaN` with an explanatory note, never a placeholder.

This notebook is a thin runner over `src/evaluate_final.run_final_evaluation()`, mirroring the pattern used in `02_train_models.ipynb`.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger
from src.evaluate_final import run_final_evaluation

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])
logger.info("05_final_evaluation notebook started (seed=%d)", seed)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## Run the full final evaluation

In order: reproduce the split (test finally used) → load frozen models + calibrators → evaluate every (model x calibration method) at full coverage → evaluate every (model x best calibration method) across all referral-coverage levels → ablation across all calibration methods → subgroup (fairness/equity) analysis → save all four tables + run metadata. All metrics use 1000 stratified bootstrap resamples for 95% CIs.

In [ ]:
results = run_final_evaluation(config=config)
print("Final evaluation complete. Tables:", list(results.keys()))

## Table 3: Main results (model x selected calibration method, full test set)

In [ ]:
table3 = results["table_3_main_results"]
display_cols = [
    "model", "calibration_method", "n_test", "auroc", "auroc_ci_lower", "auroc_ci_upper",
    "auprc", "auprc_ci_lower", "auprc_ci_upper", "brier_score", "ece",
    "calibration_intercept", "calibration_slope",
]
table3[display_cols].round(4)

## Table 4: Clinical referral thresholds (model x referral-coverage level)

In [ ]:
table4 = results["table_4_clinical_thresholds"]
display_cols = [
    "model", "calibration_method", "referral_rate", "coverage", "n_accepted", "n_referred",
    "sensitivity", "specificity", "precision", "f1_score", "balanced_accuracy", "brier_score", "ece",
]
table4[display_cols].round(4)

## Table 5: Ablation study (every calibration method, full test set)

In [ ]:
table5 = results["table_5_ablation_study"]
display_cols = [
    "model", "calibration_method", "is_selected_method", "auroc", "brier_score", "ece",
    "calibration_intercept", "calibration_slope",
]
table5[display_cols].round(4)

## Table 6: Subgroup results (post-hoc fairness/equity audit)

Rows with `note` non-empty were skipped (subgroup too small, too few events, or a single class present) — metrics are `NaN` for those rows, never approximated.

In [ ]:
table6 = results["table_6_subgroup_results"]
display_cols = [
    "model", "calibration_method", "subgroup_variable", "subgroup_level", "n", "n_events",
    "auroc", "sensitivity", "specificity", "brier_score", "ece", "note",
]
table6[display_cols].round(4)

## Summary

This is the final, single authorized evaluation of held-out test-set performance for this project. All four tables (`table_3_main_results.csv`, `table_4_clinical_thresholds.csv`, `table_5_ablation_study.csv`, `table_6_subgroup_results.csv`) are saved under `outputs/tables/`, and run metadata (seed, bootstrap count, frozen choices, software versions) is saved to `logs/final_evaluation_run_metadata.json`.

Reminder: results at any coverage/referral level remain **clinical decision support**, not autonomous diagnosis — see `src/selective_prediction.py`'s `CLINICAL_DISCLAIMER`.